In [ ]:
import sys
import subprocess

required = ["transformers", "datasets", "scipy", "pandas", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "cross-encoder/stsb-distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "test"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2"]].copy()
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())


In [ ]:
pipe_device = "mps" if device == "mps" else -1

clf = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipe_device,
    return_all_scores=False,
)

print({
    "pipeline_task": clf.task,
    "model_name": model_name,
    "pipeline_device": pipe_device,
})


In [ ]:
pairs = [
    {"text": s1, "text_pair": s2}
    for s1, s2 in zip(df["sentence1"].tolist(), df["sentence2"].tolist())
]

outputs = clf(
    pairs,
    batch_size=batch_size,
    truncation=True,
)

raw_scores = np.array([float(x["score"]) for x in outputs], dtype=np.float32)
labels_out = [str(x.get("label", "")) for x in outputs]

def parse_label_value(label):
    text = str(label).strip().upper()
    for prefix in ["LABEL_", "STAR", "SCORE_"]:
        if text.startswith(prefix):
            text = text[len(prefix):]
            break
    try:
        return float(text)
    except Exception:
        return np.nan

label_values = np.array([parse_label_value(x) for x in labels_out], dtype=np.float32)

predicted_score_0_5 = np.where(np.isfinite(label_values), label_values, raw_scores)
predicted_score_0_5 = np.clip(predicted_score_0_5, 0.0, 5.0).astype(np.float32)

if np.nanmax(raw_scores) <= 1.0 + 1e-6:
    rescaled_from_probability = (raw_scores * 5.0).astype(np.float32)
else:
    rescaled_from_probability = np.clip(raw_scores, 0.0, 5.0).astype(np.float32)

preview_df = df.copy()
preview_df["model_label"] = labels_out
preview_df["raw_model_score"] = raw_scores
preview_df["predicted_score_0_5"] = predicted_score_0_5
preview_df["rescaled_from_raw"] = rescaled_from_probability

print(preview_df[[
    "sentence1", "sentence2", "model_label", "raw_model_score",
    "predicted_score_0_5", "rescaled_from_raw"
]].head(10))


In [ ]:
score_series = pd.Series(predicted_score_0_5)
raw_series = pd.Series(raw_scores)
rescaled_series = pd.Series(rescaled_from_probability)

summary = {
    "num_examples": int(len(df)),
    "predicted_score_min": float(score_series.min()),
    "predicted_score_max": float(score_series.max()),
    "predicted_score_mean": float(score_series.mean()),
    "predicted_score_median": float(score_series.median()),
    "predicted_score_std": float(score_series.std(ddof=0)),
    "predicted_score_q05": float(score_series.quantile(0.05)),
    "predicted_score_q25": float(score_series.quantile(0.25)),
    "predicted_score_q75": float(score_series.quantile(0.75)),
    "predicted_score_q95": float(score_series.quantile(0.95)),
    "raw_score_mean": float(raw_series.mean()),
    "raw_score_std": float(raw_series.std(ddof=0)),
    "rescaled_score_mean": float(rescaled_series.mean()),
    "rescaled_score_std": float(rescaled_series.std(ddof=0)),
    "num_unique_model_labels": int(pd.Series(labels_out).nunique()),
}

runtime_seconds = time.time() - start_time

for k, v in summary.items():
    print(f"{k}: {v}")
print(f"runtime_seconds: {runtime_seconds:.2f}")


In [ ]:
print("lowest_predicted_similarity_examples")
print(preview_df.sort_values("predicted_score_0_5", ascending=True).head(10).to_string(index=False))

print("highest_predicted_similarity_examples")
print(preview_df.sort_values("predicted_score_0_5", ascending=False).head(10).to_string(index=False))

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"predicted_score_mean: {score_series.mean():.6f}")
print(f"predicted_score_std: {score_series.std(ddof=0):.6f}")
print(f"predicted_score_min: {score_series.min():.6f}")
print(f"predicted_score_max: {score_series.max():.6f}")
print(f"predicted_score_median: {score_series.median():.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
